In [11]:
import pandas as pd
import numpy as np

In [4]:
raw_df = pd.read_csv('data/search-results.csv', index_col=0)

In [7]:
# Remove duplicates
raw_df = raw_df.drop_duplicates(subset=['PaperTitle', 'DOI'])

In [ ]:
# Prepare DataFrame to store the results
columns = [
    "PaperTitle",
    "DOI",
    "Authors",
    "Abstract",
    "Publisher",
    "SemanticScholarUrl",
    "DoiUrl",
    "PublicationDate",
    "FieldOfStudy",
    "Conference-Journal",
    "PublicationTypes",
    "SearchString",
    "CitationCount",
    "SearchedFrom",
]

In [16]:
# loop through rows

for index, row in raw_df.iterrows():
    if pd.isnull(row['DOI']):
        print('No DOI for', row['PaperTitle'])
    else:
        print('DOI for', row['PaperTitle'], 'is', row['DOI'])
        # TODO: search for paper info using DOI

No DOI for FakeSpotter: A Simple Baseline for Spotting AI-Synthesized Fake Faces
No DOI for Machine Learning for Cybersecurity Cookbook
DOI for Deepfake: A Survey on Facial Forgery Technique Using Generative Adversarial Network is 10.1109/ICCS45141.2019.9065881
No DOI for Dynamic Algorithmic Service Agreements Perspective
No DOI for Release Strategies and the Social Impacts of Language Models
No DOI for The Patentability of Genetic Therapies: CAR-T and Medical Treatment Exclusions Around The World
No DOI for Teachers' and Principals' Perceptions of Antibullying Programs in a U.S. Middle School
DOI for Bringing the Bosses to International Criminal Trials: The Problems with Joint Criminal Enterprise and the “Control over the Crime” Approach As a Better Alternative is 10.58948/2331-3536.1394
No DOI for KEPASTIAN HUKUM PENGATURAN JAMINAN SOSIAL KETENAGAKERJAAN PEKERJA MIGRAN INDONESIA PASCA UNDANG-UNDANG NOMOR 18 TAHUN 2017 TENTANG PERLINDUNGAN PEKERJA MIGRAN INDONESIA
DOI for Die Diatomee

In [4]:
crossref = Crossref()

In [8]:
def extract_data(sch_results, results: list, query: str):
    for sch_paper in sch_results:
        try:
            crossref_paper = crossref.works(ids=sch_paper["externalIds"].get("DOI"))
        except Exception as e:
            crossref_paper = None

        title = sch_paper["title"]
        doi = sch_paper["externalIds"].get("DOI")

        authors = None
        # author name and affiliation
        if crossref_paper is not None:
            authors = crossref_paper.get("message").get("author")
        if authors is not None:
            for i in range(len(authors)):
                author = authors[i]
                author_name = author.get("given", "") + " " + author.get("family", "")
                affiliation = author.get("affiliation", "No Affiliation")
                affiliations = author.get("affiliation", [])
                school_names = (
                    [affil.get("name") for affil in affiliations]
                    if affiliations
                    else ["No Affiliation"]
                )
                # Create a new dictionary with only 'name' and 'affiliation'
                authors[i] = {
                    "name": author_name.strip(),
                    "affiliation": school_names,
                }
        else:
            authors = sch_paper["authors"]
            for i in range(len(sch_paper["authors"])):
                author = sch_paper["authors"][i]
                sch_paper["authors"][i] = {
                    "name": author.get("name", "No Name"),
                    "affiliation": author.get("affiliation", "No Affiliation"),
                }

        abstract = sch_paper["abstract"]
        sch_url = sch_paper["url"]
        doi_url = f"https://doi.org/{doi}"
        publication_date = sch_paper["publicationDate"]
        fields_of_study = sch_paper["fieldsOfStudy"]
        venue = sch_paper["venue"]

        # publisher
        if crossref_paper is not None:
            publisher = crossref_paper.get("message").get("publisher")
        elif doi and "arxiv" in doi.lower():
            publisher = "arXiv"
        else:
            publisher = None

        # paper type
        if crossref_paper is not None:
            paper_type = [crossref_paper.get("message").get("type")]
        else:
            paper_type = sch_paper["publicationTypes"]

        citation_count = sch_paper["citationCount"]
        # TODO: paper keywords missing
        # TODO: paper type is conference/journal for arxiv papers
        # TODO: conference-journal name mismatch with publisher, i.e., for paper with name"ChatGPT in education: A discourse analysis of worries and concerns on social media", the conference name is "International Conference on Artificial Intelligence in Education", but the publisher is "Arxiv" (becauseit queryed from arxiv), need "Springer" instead.

        new_paper = {
            "PaperTitle": title,
            "DOI": doi,
            "Authors": authors,
            "Abstract": abstract,
            "Publisher": publisher,
            "SemanticScholarUrl": sch_url,
            "DoiUrl": doi_url,
            "PublicationDate": publication_date,
            "FieldOfStudy": fields_of_study,
            "Conference-Journal": venue,
            "PublicationTypes": paper_type,
            "SearchString": query,
            "CitationCount": citation_count,
            "SearchedFrom": "Semantic Scholar",
        }
        print(new_paper)
        results.append(new_paper)

In [9]:
results_df = pd.DataFrame(results, columns=columns)
results_df.to_csv("data/initial-scrape-result.csv", index=False)

{'PaperTitle': 'More Than Just Facts: Promoting Civic Media Literacy in the Era of Outrage', 'DOI': '10.1080/0161956X.2019.1553582', 'Authors': [{'name': 'Ellen Middaugh', 'affiliation': ['San José State University, San Jose, California, USA']}], 'Abstract': 'Abstract Amid rising concerns about “fake news,” efforts have emerged to explain the spread and impact of misinformation on youth civic engagement. These efforts have focused primarily on the role of social media in exposing youth to factually inaccurate civic information and the factors that influence the ability to discern the accuracy of such information. A less explored aspect has been the impact of the rise of “outrage language,” defined as language that evokes strong emotional responses (e.g., fear, anger, disgust) that communications scholars have documented as playing a larger role in political discourse over the past few decades (Berry & Sobieraj, 2014). This article draws on three recent studies of digital media and yout

KeyboardInterrupt: 